<a href="https://colab.research.google.com/github/channelable/IHub_AIML_Training_Program/blob/main/AIML_Module_4_Proj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 4: Linear Classifiers & Gradient Descent

**Case Study: Predictive Modeling for Public Water Safety**

**Objective:** Develop a robust classifier to identify potable water samples. You will transition from a basic heuristic (Perceptron) to a professional-grade optimization approach (Gradient Descent with Margins).

# 1. Data Acquisition & Cleaning

In real-world data science, datasets are rarely perfect. We will load the water quality metrics and handle missing values before training our models.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Load the dataset from the public raw GitHub URL
url = "https://raw.githubusercontent.com/nferran/tp_aprendizaje_de_maquina_I/main/water_potability.csv"
df = pd.read_csv(url)

# Step 1: Handling Missing Values
df.fillna(df.mean(numeric_only=True), inplace=True)

# Step 2: Feature Selection & Labeling
X = df.drop('Potability', axis=1).values
y = df['Potability'].values

# Step 3: Class Label Conversion
y = np.where(y == 0, -1, 1)

# Step 4: Train-Test Split & Scaling
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Dataset Loaded: {X_train.shape[0]} training samples, {X_train.shape[1]} features.")


Dataset Loaded: 2620 training samples, 9 features.


# 2. Phase 1: The Heuristic Approach (Perceptron)

The **Perceptron** represents the earliest form of supervised learning. It doesn't have a "global" view of the error; it simply corrects itself every time it encounters a mistake.

**Task:** Implement the Perceptron Update Rule inside the training loop.

In [ ]:
class WaterPerceptron:
    def __init__(self, lr=0.01, epochs=50):
        self.lr = lr
        self.epochs = epochs
        self.w = None
        self.b = 0
        self.mistakes = []

    def fit(self, X, y):
        self.w = np.zeros(X.shape[1])
        self.b = 0
        self.mistakes = []

        for epoch in range(self.epochs):
            count = 0
            for i in range(len(y)):
                # Linear output
                prediction = np.dot(self.w, X[i]) + self.b

                # Perceptron update when y_i * f(x_i) <= 0
                if y[i] * prediction <= 0:
                    self.w += self.lr * y[i] * X[i]
                    self.b += self.lr * y[i]
                    count += 1

            self.mistakes.append(count)

    def predict(self, X):
        return np.where(np.dot(X, self.w) + self.b >= 0, 1, -1)

model_p = WaterPerceptron(lr=0.01, epochs=50)
model_p.fit(X_train, y_train)


# 3. Phase 2: Gradient Descent - Global Optimization

The Perceptron is unstable if the data isn't perfectly separable. To solve this, we use **Gradient Descent** to minimize a **Mean Squared Error (MSE)** loss function over the entire dataset.

**Task:** Implement the batch gradient calculation for weights and bias.

In [ ]:
class GDWaterClassifier:
    def __init__(self, lr=0.001, epochs=500):
        self.lr = lr
        self.epochs = epochs
        self.w = None
        self.b = 0
        self.cost_history = []

    def fit(self, X, y):
        self.w = np.zeros(X.shape[1])
        self.b = 0
        self.cost_history = []
        n = X.shape[0]

        for _ in range(self.epochs):
            # Linear output
            z = X.dot(self.w) + self.b

            # MSE gradient
            error = z - y
            dw = (1 / n) * X.T.dot(error)
            db = (1 / n) * np.sum(error)

            # Gradient descent update
            self.w -= self.lr * dw
            self.b -= self.lr * db

            # Store MSE for convergence analysis
            cost = np.mean((z - y) ** 2) / 2
            self.cost_history.append(cost)

    def predict(self, X):
        return np.where(np.dot(X, self.w) + self.b >= 0, 1, -1)

model_gd = GDWaterClassifier(lr=0.001, epochs=500)
model_gd.fit(X_train, y_train)


# 4. Phase 3: Margin Classifiers & Hinge Loss

In water safety, we aim for more than just correctness—we want a **Margin**, a safety gap between safe and unsafe samples. This is achieved using **Hinge Loss** combined with **L2 Regularization**.

The loss function is defined as:

$$
\text{Loss} = \lambda \|w\|^2_2 + \sum_{i} \max(0, 1 - y_i (w^T x_i + b))
$$

### Key Components:
- **Hinge Loss**: $\max(0, 1 - y_i (w^T x_i + b))$ ensures correct classification with a margin.
- **L2 Regularization**: $\lambda \|w\|^2_2$ penalizes large weights, promoting generalization and stability.


In [ ]:
class MarginWaterClassifier:
    def __init__(self, lr=0.001, lambda_param=0.01, epochs=500):
        self.lr = lr
        self.lambda_param = lambda_param
        self.epochs = epochs
        self.w = None
        self.b = 0

    def fit(self, X, y):
        self.w = np.zeros(X.shape[1])
        self.b = 0

        for _ in range(self.epochs):
            for i, x_i in enumerate(X):
                margin = y[i] * (np.dot(self.w, x_i) + self.b)

                if margin >= 1:
                    # Correctly classified with sufficient margin:
                    # only L2 regularization updates the weights.
                    self.w -= self.lr * (2 * self.lambda_param * self.w)
                else:
                    # Hinge-loss gradient + L2 regularization.
                    self.w -= self.lr * (
                        2 * self.lambda_param * self.w - x_i * y[i]
                    )
                    self.b -= self.lr * (-y[i])

    def predict(self, X):
        return np.where(np.dot(X, self.w) + self.b >= 0, 1, -1)

model_margin = MarginWaterClassifier(lr=0.001, lambda_param=0.01, epochs=500)
model_margin.fit(X_train, y_train)


# 5. Critical Analysis & Comparison

# 1. Convergence plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, len(model_p.mistakes) + 1), model_p.mistakes, marker='o')
axes[0].set_title("Perceptron: Mistakes per Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Number of Mistakes")
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, len(model_gd.cost_history) + 1), model_gd.cost_history)
axes[1].set_title("Gradient Descent: MSE Cost")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Cost")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 2. Test accuracy for all three models
pred_p = model_p.predict(X_test)
pred_gd = model_gd.predict(X_test)
pred_margin = model_margin.predict(X_test)

results = pd.DataFrame({
    "Model": ["Perceptron", "Gradient Descent (MSE)", "Margin Classifier (Hinge + L2)"],
    "Test Accuracy": [
        accuracy_score(y_test, pred_p),
        accuracy_score(y_test, pred_gd),
        accuracy_score(y_test, pred_margin)
    ]
})
display(results)

# 3. Safety margin
print(
    "For samples close to the decision boundary, the Margin Classifier is preferable "
    "because hinge loss explicitly encourages correctly classified samples to lie "
    "at least one unit away from the boundary, while L2 regularization controls "
    "the size of the weights."
)


# Discussion Answers

### Q1: Impact of High Learning Rate in Gradient Descent

If the learning rate is too high (for example, `1.0`), Gradient Descent can take steps that are too large. Instead of steadily moving toward the minimum of the loss function, it may overshoot the minimum repeatedly. The cost can oscillate, fail to converge, or even diverge and become extremely large. A smaller learning rate generally produces slower but more stable convergence.

### Q2: Label Conversion in Classification

The labels were converted from `{0, 1}` to `{-1, 1}` because the Perceptron and hinge-loss formulations use the signed product `y_i * f(x_i)`. With `y_i` in `{-1, 1}`, a positive product directly represents agreement between the prediction and the class, and the hinge-loss condition `y_i f(x_i) >= 1` has the same symmetric form for both classes. Using `{0, 1}` would not provide this symmetric signed-margin formulation.

### Q3: Handling Noisy Data (Water Potability Dataset)

The **MarginWaterClassifier** is the best suited of the three implemented approaches for noisy, non-perfectly-separable data. The Perceptron relies on correcting individual mistakes and does not explicitly optimize a global margin. The MSE-based Gradient Descent classifier provides global optimization, but it is not specifically designed around classification margins. The margin classifier uses hinge loss together with L2 regularization, encouraging a safety margin while limiting excessively large weights, which generally gives better robustness and generalization in noisy classification problems.
